# 🚀 مشروع مكينة صنع النماذج ثلاثية الأبعاد (Shape-E + T4 GPU)
تم التطوير بأقصى الإجراءات الاحتياطية والتثبت الذكي من المكتبات والمزامنة مع GitHub.

In [ ]:
# @title 1. التثبت الذكي من البيئة والمكتبات (بدون تكرار التنزيل)
import os
import sys
import subprocess
import torch

print("=== 1. الفحص الأولي للبيئة ===")

# 1. التحقق من تفعيل الـ GPU T4
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ خطأ: لم يتم تفعيل الـ GPU! يرجى الذهاب إلى Runtime > Change runtime type واختيار T4 GPU.")
else:
    print(f"✅ تم اكتشاف الـ GPU بنجاح: {torch.cuda.get_device_name(0)}")

# 2. دالة التثبت الذكي للمكتبات لضمان عدم التحميل مجدداً وتوفير الوقت
def install_if_missing(package_name, import_name=None, install_cmd=None):
    import_name = import_name or package_name
    try:
        __import__(import_name)
        print(f"✅ المكتبة [{import_name}] مثبتة مسبقاً.")
    except ImportError:
        print(f"🔄 المكتبة [{import_name}] غير موجودة. جاري التثبيت الاحترافي...")
        cmd = install_cmd or f"pip install {package_name} --quiet"
        subprocess.check_call(sys.executable + " -m " + cmd, shell=True)

# تفعيل التثبيتات الاحتياطية
install_if_missing("gradio", "gradio")
install_if_missing("trimesh", "trimesh")

# تثبيت Shape-E من المستودع الرسمي مباشرة مع التحقق
try:
    import shap_e
    print("✅ مكتبة [Shape-E] مثبتة ومتاحة مسبقاً.")
except ImportError:
    print("🔄 جاري تثبيت Shape-E من OpenAI (قد يستغرق دقيقة)...")
    subprocess.check_call(sys.executable + " -m pip install git+https://github.com/openai/shap-e.git --quiet", shell=True)

print("\n🎉 انتهى فحص وتثبيت المكتبات بنجاح وجميع الاحتياطات سليمة!")

In [ ]:
# @title 2. تحميل نماذج Shape-E وتحسين الأداء لـ T4
print("=== 2. تحميل نماذج ذكاء الآلة ===")
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    print("🔄 جاري تحميل أوزان Transmitter (Transmitter Weights)...")
    xm = load_model('transmitter', device=device)
    
    print("🔄 jari تحميل أوزان Text-to-3D...")
    model = load_model('text3d', device=device)
    
    print("🔄 جاري تحميل إعدادات مستند الـ Diffusion...")
    diffusion = diffusion_from_config(load_config('diffusion'))
    
    print("✅ تم تحميل جميع النماذج بنجاح واستقرار عالي على كرت T4!")
except Exception as e:
    print(f"⚠️ حدث خطأ أثناء تحميل النماذج: {e}")

In [ ]:
# @title 3. دالة الحفظ التلقائي والمزامنة مع GitHub عبر الـ API
import os
import requests
import base64

def upload_to_github(file_path, repo_name, github_token, commit_message="Add 3D Model"):
    if not github_token or not repo_name:
        return "⚠️ لم يتم تزويد بيانات GitHub كاملة، تم الاعتماد على التحميل المحلي المباشر."
    
    try:
        filename = os.path.basename(file_path)
        url = f"https://api.github.com/repos/{repo_name}/contents/{filename}"
        
        with open(file_path, "rb") as file:
            content = base64.b64encode(file.read()).decode("utf-8")
            
        headers = {
            "Authorization": f"token {github_token}",
            "Accept": "application/vnd.github.v3+json"
        }
        
        # التثبت الاحتياطي مما إذا كان الملف مرفوعاً مسبقاً لتحديثه
        get_res = requests.get(url, headers=headers)
        sha = get_res.json().get("sha") if get_res.status_code == 200 else None
        
        data = {
            "message": commit_message,
            "content": content
        }
        if sha:
            data["sha"] = sha
            
        res = requests.put(url, headers=headers, json=data)
        
        if res.status_code in [200, 201]:
            return f"✅ تم الرفع بنجاح ومزامنته مع GitHub: {res.json()['content']['html_url']}"
        else:
            return f"❌ فشل الرفع لـ GitHub. كود الخطأ: {res.status_code} - {res.text}"
            
    except Exception as e:
        return f"❌ خطأ غير متوقع أثناء الرفع لـ GitHub: {str(e)}"

In [ ]:
# @title 4. إطلاق واجهة Gradio المستقلة للتوليد والعرض ثلاثي الأبعاد
import gradio as gr
from shap_e.util.notebooks import decode_latent_mesh
import time

def process_generation(prompt, guidance_scale, steps, use_github, github_repo, github_token):
    if not prompt.strip():
        return None, "⚠️ يرجى إدخال وصف نصي أولاً."
        
    output_filename = f"model_{int(time.time())}.ply"
    status_msg = ""
    
    try:
        print(f"🎬 جاري بدء عملية هندسة المجسم: {prompt}")
        # احتياطياً: إفراغ الذاكرة للـ GPU لمنع نفادها
        torch.cuda.empty_cache()
        
        # توليد الأبعاد المخفية المتغيرة (Latents)
        latents = sample_latents(
            batch_size=1,
            model=model,
            diffusion=diffusion,
            guidance_scale=guidance_scale,
            model_kwargs=dict(texts=[prompt]),
            progress=True,
            clip_denoised=True
        )
        
        # تحويل ويفك تشفير المجسم إلى شبكة هندسية 3D
        with torch.no_grad():
            mesh = decode_latent_mesh(xm, latents[0]).triangulate()
            with open(output_filename, 'wb') as f:
                mesh.write_ply(f)
                
        status_msg = "✅ تم التوليد بنجاح وحفظ الملف في خادم كولاب محلياً."
        
        # مزامنة الملف مع مستودع GitHub تلقائياً إذا رغب المستخدم
        if use_github and github_token and github_repo:
            github_status = upload_to_github(output_filename, github_repo, github_token, f"Add 3D model for: {prompt}")
            status_msg += f"\n{github_status}"
        elif use_github:
            status_msg += "\n⚠️ لم يتم الرفع لـ GitHub بسبب نقص في معطيات الإعدادات (Token أو اسم المستودع)."
            
        return output_filename, status_msg
        
    except Exception as e:
        torch.cuda.empty_cache()
        return None, f"❌ حدث خطأ برمجي غير متوقع: {str(e)}"

# بناء الواجهة التفاعلية المستقلة
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 مكينة صنع النماذج ثلاثية الأبعاد (Shape-E + T4 GPU)")
    gr.Markdown("أدخل وصفاً للمجسم باللغة الإنجليزية لتصميمه فوراً، مع إمكانية عرضه والتحكم فيه، ثم حفظه آلياً.")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(label="📝 وصف النموذج (Prompt)", placeholder="مثال: a golden crown, a small coffee cup...", lines=3)
            
            with gr.Accordion("⚙️ إعدادات المحرك المتقدمة", open=False):
                guidance_slider = gr.Slider(minimum=1.0, maximum=25.0, value=15.0, step=0.5, label="Guidance Scale")
                steps_slider = gr.Slider(minimum=15, maximum=100, value=64, step=1, label="Steps")
            
            with gr.Accordion("🌐 إعدادات المزامنة والرفع لـ GitHub", open=False):
                enable_github = gr.Checkbox(label="تفعيل الحفظ التلقائي في المستودع", value=False)
                repo_input = gr.Textbox(label="اسم المستودع (Username/Repository)", placeholder="مثال: my_git_user/models_repo")
                token_input = gr.Textbox(label="GitHub Token (ghp_...)", type="password")
                
            generate_btn = gr.Button("🎨 ابدأ الإنتاج الآن", variant="primary")
            
        with gr.Column(scale=1):
            viewer_output = gr.Model3D(label="📦 نافذة استعراض المجسم 3D المعماري")
            status_output = gr.Textbox(label="📊 لوحة المراقبة والتحقق من العمليات", interactive=False)
            
    generate_btn.click(
        fn=process_generation,
        inputs=[prompt_input, guidance_slider, steps_slider, enable_github, repo_input, token_input],
        outputs=[viewer_output, status_output]
    )

# التفعيل برابط عام مستقل وتهيئة ميكانيكية الـ queue لمنع حدوث التجميد
demo.queue().launch(share=True, debug=True)

In [ ]:
# @title 5. فحص التثبت النهائي والتلقائي من سلامة عمل المحرك (Automated Sanity Check)
print("=== 5. فحص التثبت النهائي للمشروع ===")
try:
    print("🔄 جاري إجراء اختبار توليد ذاتي وسريع للتأكد من ربط التوابع بالذاكرة...")
    test_latents = sample_latents(
        batch_size=1,
        model=model,
        diffusion=diffusion,
        guidance_scale=5.0,
        model_kwargs=dict(texts=["a simple ring"]),
        progress=False,
        clip_denoised=True
    )
    
    with torch.no_grad():
        test_mesh = decode_latent_mesh(xm, test_latents[0]).triangulate()
        test_mesh.write_ply("sanity_check_output.ply")
        
    if os.path.exists("sanity_check_output.ply"):
        print("🏁 [تثبت نهائي ناجح]: تم فحص المشروع بالكامل تلقائياً وهو جاهز 100% لطلبات المستخدم!")
        os.remove("sanity_check_output.ply")
    else:
        print("⚠️ تم التحقق برمجياً ولكن لم يتم العثور على الملف الصامت، يرجى إعادة مراجعة صلاحيات التخزين.")
except Exception as e:
    print(f"❌ فشل فحص التثبت النهائي وتكامل الملفات. السبب: {e}")